# SheetSage-Infer - Music Transcription to Lead Sheets

This notebook demonstrates how to use SheetSage-Infer for transcribing music audio to lead sheets (melody + chord symbols).

**Features:**
- Transcribe audio to lead sheets automatically
- Extract melody and chord symbols
- Multiple export formats (LilyPond, MIDI, PDF)
- CPU (handcrafted features) or GPU (Jukebox embeddings) support
- High-quality music transcription using hierarchical transformer architecture


## 1. Installation


In [ ]:
# Install from PyPI
!pip install -q openmirlab-sheetsage-infer

# Optional: Install LilyPond for PDF generation (Linux/Colab)
# !apt-get -qq install -y lilypond

# Optional: Install YouTube support
# !pip install -q "openmirlab-sheetsage-infer[youtube]"


In [ ]:
# Verify installation
import torch
from sheetsage.infer import sheetsage

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Prepare Directories

Create directories for input audio files and output files.


In [ ]:
# Create directories
!mkdir -p input_songs
!mkdir -p outputs


## 3. Upload Your Audio File

Upload a `.wav`, `.mp3`, or other audio file to transcribe to a lead sheet.


In [ ]:
# Option 1: Upload from your computer
from google.colab import files

print("Upload your audio file (.wav, .mp3, etc.):")
uploaded = files.upload()

# Move uploaded file to input folder
for filename in uploaded.keys():
    !mv "{filename}" input_songs/
    print(f"Moved {filename} to input_songs/")


In [ ]:
# Option 2: Download from URL
# Uncomment and modify the URL below:
# !wget -q -O input_songs/audio.mp3 "https://example.com/audio.mp3"


In [ ]:
# Check input files
!ls -lh input_songs/


## 4. Run Music Transcription

Select your preferred method and run the transcription:


In [ ]:
#@title Select Inference Options { display-mode: "form" }
#@markdown Choose inference method and feature extraction:

inference_method = "Python API" #@param ["Python API", "CLI (Command Line)"]
feature_method = "Handcrafted (CPU)" #@param ["Handcrafted (CPU)", "Jukebox (GPU)"]
segment_start_hint = 30 #@param {type:"number"}
segment_end_hint = 60 #@param {type:"number"}
beats_per_minute_hint = 120 #@param {type:"number"}

use_jukebox = (feature_method == "Jukebox (GPU)")

print(f"Selected method: {inference_method}")
print(f"Feature extraction: {feature_method}")
if use_jukebox:
    print("⚠️  Note: Jukebox requires GPU with ≥12GB VRAM")


In [ ]:
#@title Run Music Transcription { display-mode: "form" }
#@markdown Click the play button to run transcription with your selected method.

from pathlib import Path
from sheetsage.infer import sheetsage
from sheetsage.utils import engrave
from sheetsage.align import create_beat_to_time_fn

if inference_method == "CLI (Command Line)":
    # ============================================
    # Option A: CLI (Command Line)
    # ============================================
    print("Running with CLI...\n")

    input_folder = Path("input_songs")
    for audio_file in input_folder.glob("*.*"):
        if audio_file.suffix.lower() in ['.wav', '.mp3', '.flac', '.m4a', '.ogg']:
            output_file = Path("outputs") / f"{audio_file.stem}"
            cmd = f'python -m sheetsage.infer "{audio_file}" -o "{output_file}"'
            if use_jukebox:
                cmd += " --use_jukebox"
            if segment_start_hint:
                cmd += f" --segment_start_hint {segment_start_hint}"
            if segment_end_hint:
                cmd += f" --segment_end_hint {segment_end_hint}"
            if beats_per_minute_hint:
                cmd += f" --beats_per_minute_hint {beats_per_minute_hint}"
            !{cmd}

else:
    # ============================================
    # Option B: Python API
    # ============================================
    print("Running with Python API...\n")

    input_folder = Path("input_songs")
    output_folder = Path("outputs")
    output_folder.mkdir(exist_ok=True)

    for audio_path in input_folder.glob("*.*"):
        if audio_path.suffix.lower() not in ['.wav', '.mp3', '.flac', '.m4a', '.ogg']:
            continue
            
        print(f"Processing: {audio_path.name}")

        # Run transcription
        lead_sheet, segment_beats, segment_beats_times = sheetsage(
            str(audio_path),
            use_jukebox=use_jukebox,
            segment_start_hint=segment_start_hint if segment_start_hint > 0 else None,
            segment_end_hint=segment_end_hint if segment_end_hint > 0 else None,
            beats_per_minute_hint=beats_per_minute_hint if beats_per_minute_hint > 0 else None,
        )

        # Export to LilyPond
        lily_code = lead_sheet.as_lily()
        lily_path = output_folder / f"{audio_path.stem}.ly"
        with open(lily_path, 'w', encoding='utf-8') as f:
            f.write(lily_code)
        print(f"  Saved LilyPond: {lily_path}")

        # Export to MIDI
        beat_to_time_fn = create_beat_to_time_fn(segment_beats, segment_beats_times)
        midi_bytes = lead_sheet.as_midi(beat_to_time_fn)
        midi_path = output_folder / f"{audio_path.stem}.mid"
        with open(midi_path, 'wb') as f:
            f.write(midi_bytes)
        print(f"  Saved MIDI: {midi_path}")

        # Generate PDF (if LilyPond is available)
        try:
            pdf_bytes = engrave(lily_code, out_format='pdf')
            pdf_path = output_folder / f"{audio_path.stem}.pdf"
            with open(pdf_path, 'wb') as f:
                f.write(pdf_bytes)
            print(f"  Saved PDF: {pdf_path}")
        except Exception as e:
            print(f"  PDF generation skipped: {e}")

    print("\nDone!")


## 5. Check Output Files


In [ ]:
# Check output files
!ls -lh outputs/


## 6. Preview Results


In [ ]:
#@title Preview Transcription Results { display-mode: "form" }
#@markdown This cell displays lead sheet info and previews the transcription.

from pathlib import Path
import pretty_midi
import IPython.display as ipd

output_dir = Path("outputs")

# Find and display all output files
for midi_file in sorted(output_dir.glob("*.mid")):
    print(f"\n{'='*60}")
    print(f"File: {midi_file.stem}")
    print(f"{'='*60}")

    # Load and display MIDI info
    try:
        midi_data = pretty_midi.PrettyMIDI(str(midi_file))
        print(f"Duration: {midi_data.get_end_time():.2f} seconds")
        print(f"Tempo: {midi_data.estimate_tempo():.1f} BPM")
        print(f"Instruments: {len(midi_data.instruments)}")

        total_notes = sum(len(inst.notes) for inst in midi_data.instruments)
        print(f"Total notes: {total_notes}")

        for inst in midi_data.instruments:
            if inst.is_drum:
                inst_name = "Drums"
            else:
                inst_name = pretty_midi.program_to_instrument_name(inst.program)
            print(f"  - {inst_name}: {len(inst.notes)} notes")
    except Exception as e:
        print(f"Error loading MIDI: {e}")

    # Display LilyPond code if available
    lily_file = output_dir / f"{midi_file.stem}.ly"
    if lily_file.exists():
        print(f"\nLilyPond notation preview (first 500 chars):")
        with open(lily_file, 'r', encoding='utf-8') as f:
            lily_content = f.read()
            print(lily_content[:500])
            if len(lily_content) > 500:
                print("...")

    # Display PDF if available
    pdf_file = output_dir / f"{midi_file.stem}.pdf"
    if pdf_file.exists():
        print(f"\nPDF available: {pdf_file.name}")
        # Display PDF inline
        from IPython.display import IFrame
        display(IFrame(str(pdf_file), width=800, height=600))


In [ ]:
# Download all output files as a zip
!zip -r outputs.zip outputs/

from google.colab import files
files.download("outputs.zip")


---

## Feature Comparison

| Feature Method | Speed | Quality | GPU Required | Best For |
|----------------|-------|---------|--------------|----------|
| Handcrafted (CPU) | ~1-5 sec/min | Good | No | Fast transcription, CPU-only environments |
| Jukebox (GPU) | ~30-60 sec/min | Higher | Yes (≥12GB VRAM) | Best quality, when GPU is available |

**Note**: Jukebox features provide higher quality transcription but require a GPU with at least 12GB VRAM. Handcrafted features work on CPU and are much faster.

See the [GitHub repository](https://github.com/openmirlab/sheetsage-infer) for more information and examples.
